In [ ]:
#KOMÓRKA 1: POBIERANIE I WCZYTYWANIE
!pip install -q pyspark kagglehub gradio

from pyspark.sql import SparkSession
import kagglehub
import os

# 1. Uruchomienie Sparka
spark = SparkSession.builder \
    .appName("FlightDelayAnalysis") \
    .config("spark.ui.port", "4050") \
    .getOrCreate()

print("SparkSession created successfully!")

# 2. Pobranie danych przez KaggleHub
path = kagglehub.dataset_download("williamparker20/flight-ontime-reporting-with-weather")
print(f"Dataset downloaded to: {path}")

# 3. Wczytanie danych do Sparka
df = spark.read.option("recursiveFileLookup", "true") \
               .csv(path, header=True, inferSchema=True)

print("Spark DataFrame loaded successfully!")
print("-" * 30)
df.printSchema()

SparkSession created successfully!


100%|██████████| 157M/157M [00:01<00:00, 118MB/s]

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/datasets/williamparker20/flight-ontime-reporting-with-weather/versions/1
Spark DataFrame loaded successfully!
------------------------------
root
 |-- Time: timestamp (nullable = true)
 |-- Origin: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- Carrier: string (nullable = true)
 |-- Cancelled: boolean (nullable = true)
 |-- CancellationReason: string (nullable = true)
 |-- Delayed: boolean (nullable = true)
 |-- DepDelayMinutes: double (nullable = true)
 |-- CarrierDelay: double (nullable = true)
 |-- WeatherDelay: double (nullable = true)
 |-- NASDelay: double (nullable = true)
 |-- SecurityDelay: double (nullable = true)
 |-- LateAircraftDelay: double (nullable = true)
 |-- Temperature: double (nullable = true)
 |-- Feels_Like_Temperature: double (nullable = true)
 |-- Altimeter_Pressure: double (nullable = true)
 |-- Sea_Level_Pressure: double (nullable = true)
 |-- Visibility: double (nullable = true)
 |-- Wind_Speed:

In [ ]:
#KOMÓRKA 2: ZAAWANSOWANE PRZETWARZANIE (FEATURE ENGINEERING)
from pyspark.sql.functions import col, when, month, hour, isnan, count
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

# 1. Wybór kolumn (DODAJEMY KLUCZOWE CECHY POGODOWE!)
# Dodajemy: 'Wind_Gust' (porywy) i 'Ice_Accretion_3hr' (lód)
cols_needed = ["DepDelayMinutes", "Time", "Origin", "Temperature",
               "Wind_Speed", "Precipitation", "Visibility",
               "Wind_Gust", "Ice_Accretion_3hr"]

# Selekcja
df_selected = df.select(cols_needed)

# 2. Obsługa brakujących danych
# Jeśli nie ma danych o porywach wiatru/lodzie, zakładamy 0
df_clean = df_selected.na.fill(0.0, ["Wind_Gust", "Ice_Accretion_3hr"]) \
                      .dropna() 

# 3. Tworzenie Etykiety i Czasu
df_prepared = df_clean.withColumn("Label", when(col("DepDelayMinutes") > 15, 1.0).otherwise(0.0)) \
                      .withColumn("Month", month(col("Time"))) \
                      .withColumn("Hour", hour(col("Time")))

# BALANSOWANIE KLAS
# ile jest opóźnionych vs o czasie
total_count = df_prepared.count()
delayed_count = df_prepared.filter(col("Label") == 1.0).count()
on_time_count = total_count - delayed_count
balance_ratio = on_time_count / delayed_count

print(f"Stosunek klas: 1 opóźniony na {balance_ratio:.2f} o czasie.")
print("Dodaję wagę do danych, aby model skupił się na opóźnieniach...")

# Dodajemy kolumnę 'classWeight' - wiersze z opóźnieniem dostają wyższą wagę
df_weighted = df_prepared.withColumn("classWeight",
                                     when(col("Label") == 1.0, balance_ratio).otherwise(1.0))

# 4. Pipeline do Cech (Dodajemy OneHotEncoder dla Lotnisk)
indexer = StringIndexer(inputCol="Origin", outputCol="OriginIndex")
encoder = OneHotEncoder(inputCols=["OriginIndex"], outputCols=["OriginVec"]) # Lepsze dla Random Forest

assembler = VectorAssembler(
    inputCols=["OriginVec", "Month", "Hour", "Temperature",
               "Wind_Speed", "Wind_Gust", "Precipitation",
               "Visibility", "Ice_Accretion_3hr"],
    outputCol="features"
)

# Podział danych
train_data, test_data = df_weighted.randomSplit([0.8, 0.2], seed=42)

print("Dane gotowe do zaawansowanego treningu.")


Stosunek klas: 1 opóźniony na 3.70 o czasie.
Dodaję wagę do danych, aby model skupił się na opóźnieniach...
Dane gotowe do zaawansowanego treningu.


In [ ]:
#KOMÓRKA 3: TRENOWANIE Z WAGAMI I NOWYMI CECHAMI
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# 1. Konfiguracja modelu
rf = RandomForestClassifier(labelCol="Label",
                            featuresCol="features",
                            weightCol="classWeight", # <--- Użycie wag
                            numTrees=40,             # Więcej drzew
                            maxDepth=10,             # Głębsza analiza
                            maxBins=64)              # Lepsze rozróżnianie lotnisk

# 2. Pipeline
pipeline = Pipeline(stages=[indexer, encoder, assembler, rf])

# 3. Trenowanie
print("Rozpoczynam trenowanie modelu zbalansowanego...")
model = pipeline.fit(train_data)
print("Model wytrenowany!")

# 4. Ewaluacja
predictions = model.transform(test_data)
evaluator = BinaryClassificationEvaluator(labelCol="Label")
auc = evaluator.evaluate(predictions)

print(f"\n>>> NOWY WYNIK (AUC): {auc:.4f} <<<")

# 5. Sprawdzenie ważności
rf_model = model.stages[-1]
importances = rf_model.featureImportances

print("\nWażność cech (indeksy):")
print(importances)
print("(Jeśli widzisz wysokie wartości na dalszych pozycjach, to znaczy że Porywy Wiatru i Lód zaczęły działać!)")

Rozpoczynam trenowanie modelu zbalansowanego...
Model wytrenowany!

>>> NOWY WYNIK (AUC): 0.6725 <<<

Ważność cech (indeksy):
(37,[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35],[0.003901810350367153,0.01923552425360128,0.0010223681693936446,0.0006811063503927978,0.0013378677344525502,0.0008203677218874266,0.004961098852980221,0.008315322624309288,0.0002678149123491141,0.0070410391222107665,0.00018519347002099118,0.0023081983212639,0.004313292736910336,0.00014240987197378774,0.00011594444481295195,0.0002457828002153049,0.001831582830876637,0.0013703313887650898,0.0040199444253157975,0.00026535572011286755,0.00012530665394238242,0.00013782914563898444,0.002010028792013626,0.013151639789528366,5.458155725908709e-05,0.00012307280326625958,5.494416073584142e-05,0.009594617820499834,0.00010432527930204513,0.06555090434793057,0.5015702996029903,0.09291027698583006,0.021211488265336653,0.01745865479572348,0.17569549815929889,0.03786417573849

In [ ]:
#KOMÓRKA 4 (ZAKTUALIZOWANA O NOWE CECHY)
import gradio as gr
from pyspark.sql import Row

airport_labels = model.stages[0].labels
airport_labels.sort()

def predict_flight_delay_advanced(airport, month_val, hour_val, temp, wind_spd, wind_gust, precip, vis, ice):
    try:
        user_data = spark.createDataFrame([
            Row(
                Origin=airport,
                Month=int(month_val),
                Hour=int(hour_val),
                Temperature=float(temp),
                Wind_Speed=float(wind_spd),
                Wind_Gust=float(wind_gust),      
                Precipitation=float(precip),
                Visibility=float(vis),
                Ice_Accretion_3hr=float(ice)     
            )
        ])

        prediction = model.transform(user_data)
        probs = prediction.select("probability").collect()[0][0]
        prob_delay = probs[1]

        if prob_delay > 0.5:
            return f"⚠️ RYZYKO OPÓŹNIENIA! (Prawdopodobieństwo: {prob_delay:.1%})\nPrzyczyną mogą być porywy wiatru lub lód."
        else:
            return f"✅ LOT PLANOWY (Ryzyko: {prob_delay:.1%})"

    except Exception as e:
        return f"Błąd: {str(e)}"

iface = gr.Interface(
    fn=predict_flight_delay_advanced,
    inputs=[
        gr.Dropdown(choices=airport_labels, label="Lotnisko", value="JFK"),
        gr.Slider(1, 12, label="Miesiąc", value=1),
        gr.Slider(0, 23, label="Godzina", value=17),
        gr.Slider(-10, 100, label="Temperatura (F)", value=32),
        gr.Slider(0, 50, label="Wiatr Średni (mph)", value=10),
        gr.Slider(0, 80, label="PORYWY Wiatru (mph) [Kluczowe!]", value=15), 
        gr.Slider(0, 5, label="Opady (cale)", value=0),
        gr.Slider(0, 10, label="Widoczność", value=10),
        gr.Slider(0, 5, label="Oblodzenie (mm) [Kluczowe!]", value=0)         
    ],
    outputs="text",
    title="System Predykcji 2.0 (Enhanced Features)",
    description="Model uwzględnia teraz porywy wiatru, oblodzenie i balans klasy."
)

iface.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bc3a62150fa91aa8d3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
